# 10a NSD (Algonauts 2023), converted for the fMRI analyses

**Kaggle:** CPU, Internet on. **Inputs:** none. **Secret:** `ALGONAUTS_DRIVE_URL`, the Google Drive folder link the
Algonauts organisers give after you accept NSD's terms.

One person at a time: download, convert, delete. Writes float16 fMRI per hemisphere, NSD image ids, region files,
each stimulus image once (256 x 256) and NSD's noise ceilings for the challenge vertices to this notebook's private
output. Structure only: no statistic of the responses is computed, so NSD stays unseen until the prespecified test.
NSD's terms apply to the output.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '10a-nsddata', 'cpu-build'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"image_size": 256, "jpeg_quality": 95, "nc_trials": 3, "beta_version": "betas_fithrf_GLMdenoise_RR"}')

import sys
import numpy as np
from cross_subject_encoding.data import nsd
from cross_subject_encoding.utils import config as C

In [ ]:
import re
import shutil
import subprocess
import urllib.request
import zipfile
from PIL import Image
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "nibabel"], check=True)
import gdown
import nibabel as nib

TMP = Path("/kaggle/tmp/algonauts")
TMP.mkdir(parents=True, exist_ok=True)
OUT = C.PROJECT_ROOT / "nsd"
(OUT / "images").mkdir(parents=True, exist_ok=True)
DRIVE_URL = C.ALGONAUTS_DRIVE_URL or os.environ.get("ALGONAUTS_DRIVE_URL", "")
if not DRIVE_URL and IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        DRIVE_URL = UserSecretsClient().get_secret("ALGONAUTS_DRIVE_URL").strip()
    except Exception:
        pass
if not DRIVE_URL:
    raise RuntimeError("No Algonauts 2023 data link: request the data at algonauts.csail.mit.edu (NSD terms apply), then "
                       "put the Google Drive folder link you receive in the ALGONAUTS_DRIVE_URL environment variable "
                       "or, on Kaggle, in a Secret of that name (Add-ons, Secrets). Never commit the link.")
listing = gdown.download_folder(DRIVE_URL, output=str(TMP), skip_download=True, quiet=True)
items = sorted((f.path, f.id, Path(f.local_path)) for f in listing)
print("Drive folder:")
for path, file_id, _ in items:
    print("  ", path, file_id)
# The folder holds train_data/subjNN.zip (the full training split, about 4.6 GB) and test_data/subjNN.zip (test images
# only, about 70 MB) per subject: only the train_data zips are used. Copies are still tried in path order
# and the first that unzips into the expected structure is kept; the copy used and any rejected one go into the report
candidates = {}
for path, file_id, local in items:
    if path.endswith(".zip") and "test_data" not in path:
        candidates.setdefault("subj%02d" % int(re.search(r"subj0?(\d)", Path(path).name).group(1)), []).append((path, file_id, local))
if sorted(candidates) != [f"subj{i:02d}" for i in range(1, 9)]:
    raise RuntimeError(f"expected zips for subj01..subj08, found {sorted(candidates)}")

def fetch(subj):
    rejected = []
    for k, (path, file_id, local) in enumerate(candidates[subj]):
        local = TMP / f"{subj}_{k}.zip"
        for wait in (0, 60, 300, 900, None):  # Google Drive refuses popular files for a while ("too many accesses")
            try:
                gdown.download(id=file_id, output=str(local), quiet=True)
                break
            except Exception as exc:
                if wait is None:
                    raise
                print(f"  {subj}: download refused ({type(exc).__name__}); retrying in {wait or 30} s")
                time.sleep(wait or 30)
        mb = local.stat().st_size / 1e6
        work = TMP / subj
        shutil.rmtree(work, ignore_errors=True)
        try:
            with zipfile.ZipFile(local) as z:
                z.extractall(work)
            found = (nsd.image_ids(nsd.find_dir(work, "training_images")), nsd.image_ids(nsd.find_dir(work, "test_images")))
            local.unlink()
            return work, found, {"used": path, "zip_mb": round(mb, 1), "rejected": rejected}
        except Exception as exc:
            rejected.append(f"{path} ({mb:.0f} MB): {type(exc).__name__}: {exc}")
            local.unlink()
    raise RuntimeError(f"{subj}: no usable zip among {rejected}")

report, saved, ids = {"subjects": {}}, set(), {}
for subj in sorted(candidates):
    t = time.monotonic()
    work, ((tr_ids, tr_paths), (te_ids, te_paths)), source = fetch(subj)
    ids[subj] = {"train": tr_ids, "test": te_ids}
    np.save(OUT / f"{subj}_train_nsd_ids.npy", tr_ids)
    np.save(OUT / f"{subj}_test_nsd_ids.npy", te_ids)
    rec = {**source, "train_images": len(tr_ids), "test_images": len(te_ids)}
    for hemi in ("lh", "rh"):
        x = np.load(nsd.find_one(work, f"{hemi}_training_fmri.npy"))
        if x.ndim != 2 or len(x) != len(tr_ids):
            raise ValueError(f"{subj} {hemi}: fMRI shape {x.shape} against {len(tr_ids)} images")
        h, err = nsd.to_float16(x)
        np.save(OUT / f"{subj}_{hemi}_fmri.npy", h)
        rec[f"{hemi}_vertices"], rec[f"{hemi}_float16_max_error"] = int(x.shape[1]), round(err, 5)
        del x, h
    roi_src = nsd.find_dir(work, "roi_masks")
    roi_out = OUT / f"{subj}_roi"
    shutil.copytree(roi_src, roi_out)
    rois = {}
    for f in sorted(roi_out.glob("mapping_*.npy")):
        cls = f.stem.removeprefix("mapping_")
        labels = np.load(f, allow_pickle=True).item()
        for hemi in ("lh", "rh"):
            m = roi_out / f"{hemi}.{cls}_challenge_space.npy"
            if m.exists():
                v = np.load(m)
                rois.update({f"{hemi}.{labels[k]}": int((v == k).sum()) for k in labels if k != 0 and (v == k).any()})
    rec["roi_vertices"] = rois
    # NSD's noise ceilings for the challenge vertices (public AWS bucket; quality only, no response statistic)
    for hemi in ("lh", "rh"):
        try:
            url = f"{C.NSD_AWS}/nsddata_betas/ppdata/{subj}/fsaverage/{FIXED['beta_version']}/{hemi}.ncsnr.mgh"
            dst = TMP / f"{subj}_{hemi}.ncsnr.mgh"
            urllib.request.urlretrieve(url, dst)
            ncsnr = np.asarray(nib.load(str(dst)).get_fdata()).squeeze()
            mask = np.load(roi_out / f"{hemi}.all-vertices_fsaverage_space.npy").astype(bool)
            sel = ncsnr[mask]
            if len(sel) != rec[f"{hemi}_vertices"]:
                raise ValueError(f"{len(sel)} masked fsaverage vertices against {rec[f'{hemi}_vertices']} challenge vertices")
            nc = nsd.noise_ceiling(sel, FIXED["nc_trials"])
            np.save(OUT / f"{subj}_{hemi}_noise_ceiling.npy", nc.astype(np.float16))
            rec[f"{hemi}_noise_ceiling"] = {"median": round(float(np.median(nc)), 1), "share_above_10": round(float((nc > 10).mean()), 3)}
        except Exception as exc:  # reported, not fatal: the conversion itself does not depend on it
            rec[f"{hemi}_noise_ceiling"] = f"failed: {type(exc).__name__}: {exc}"
    for p, i in zip(list(tr_paths) + list(te_paths), np.r_[tr_ids, te_ids]):
        if int(i) not in saved:
            with Image.open(p) as im:
                im.convert("RGB").resize((FIXED["image_size"],) * 2, Image.BICUBIC).save(
                    OUT / "images" / f"nsd-{int(i):05d}.jpg", quality=FIXED["jpeg_quality"])
            saved.add(int(i))
    shutil.rmtree(work)
    report["subjects"][subj] = rec
    print(subj, rec)

for path, file_id, local in items:
    if not path.endswith(".zip"):
        gdown.download(id=file_id, output=str(OUT / Path(path).name), quiet=True)

In [ ]:
shared_train = nsd.shared_ids({s: v["train"] for s, v in ids.items()})
shared_any = nsd.shared_ids({s: np.r_[v["train"], v["test"]] for s, v in ids.items()})
np.save(OUT / "shared_train_nsd_ids.npy", shared_train)
report["shared_training_images_all_8"] = int(len(shared_train))
report["shared_images_all_8_train_or_test"] = int(len(shared_any))
report["unique_images_saved"] = len(saved)
report["output_gb"] = round(sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file()) / 1e9, 2)
report["fixed"] = FIXED
(C.METRICS_DIR / "10a_report.json").write_text(json.dumps(report, indent=2))
print("10A_REPORT_JSON " + json.dumps(report))  # the full report also goes to the log, since this output holds about 70,000 files